In [1]:
from google.colab import auth
import gspread
import pandas as pd

# Authenticate user
auth.authenticate_user()

# Connect to Google Sheets
from google.colab import auth
import google.auth
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

In [2]:
# Open the spreadsheet by its name
spreadsheet = gc.open('Mentorship_database')

# Select the staging worksheet
raw_sheet = spreadsheet.worksheet('raw_data')

# Convert the sheet data into a Pandas DataFrame
data = raw_sheet.get_all_records()
df = pd.DataFrame(data)

# Take a quick look at the first few rows
df.head()

,Session_ID,Date,Cohort_Quarter,Student_ID,Mentor_ID,STEM_Field,Department_Owner,Budget_Allocated,Budget_Spent,Attendance_Status,Satisfaction_Score,Outcome_Tracked
0,SES-2026-001,2026-11-24,Q4-2026,STU-118,MNT-524,Robotics,Operations,300,294,Attended,2,Landed Internship
1,SES-2026-002,2026-10-30,Q4-2026,STU-165,MNT-520,Robotics,Curriculum,900,892,Attended,2,Landed Internship
2,SES-2026-003,2026-11-29,Q4-2026,STU-101,MNT-525,Software Engineering,Outreach,1500,1361,Attended,4,Retained in STEM
3,SES-2026-004,2026-06-24,Q2-2026,STU-112,MNT-513,Robotics,Finance,700,661,Attended,3,Completed Project
4,SES-2026-005,2026-06-26,Q2-2026,STU-169,MNT-504,Data Science,Operations,1200,1133,Absent,,Ongoing


In [3]:
import numpy as np
#Adding some demography
# Define your localized options
locations = ['Pretoria', 'Joburg', 'Mamelodi', 'Soshanguve', 'Winterveld']
universities = ['UP', 'UJ', 'WITS', 'UCT']
genders = ['Male', 'Female', 'Non-binary']
races = ['African', 'White', 'Coloured', 'Indian', 'Asian']

# Set a random seed so the results are reproducible
np.random.seed(42)

# Programmatically generate and assign the new columns to your existing DataFrame
df['Student_Gender'] = np.random.choice(genders, size=len(df))
df['Mentor_Gender'] = np.random.choice(genders, size=len(df))
df['Student_Location'] = np.random.choice(locations, size=len(df))
df['Mentor_School'] = np.random.choice(universities, size=len(df))
df['Student_Race'] = np.random.choice(races, size=len(df))
df['Mentor_Race'] = np.random.choice(races, size=len(df))

print("Demographic data successfully injected!")
df[['Student_Location', 'Mentor_School', 'Student_Race']].head()

Demographic data successfully injected!


,Student_Location,Mentor_School,Student_Race
0,Soshanguve,UCT,Coloured
1,Joburg,UP,Asian
2,Joburg,UP,Asian
3,Mamelodi,UJ,Asian
4,Pretoria,WITS,Asian


In [4]:
# 1. Standardize the Date column
# This forces all messy dates into a uniform standard date format
df['Date'] = pd.to_datetime(df['Date'], errors='coerce').dt.strftime('%Y-%m-%d')

# 2. Handle missing satisfaction scores
# If a student attended but score is missing, we fill it with the median score (e.g., 4)
# so it doesn't skew our dashboard aggregations negatively.
#Convert Score from text to numerical
df['Satisfaction_Score'] = pd.to_numeric(df['Satisfaction_Score'], errors='coerce')
median_score = df['Satisfaction_Score'].median()
df['Satisfaction_Score'] = df['Satisfaction_Score'].fillna(median_score)

# 3. Handle Budget Overruns
# create a flag column
# called 'Budget_Overrun' (True/False) so the Finance Department can easily filter for leaks.
df['Budget_Overrun'] = df['Budget_Spent'] > df['Budget_Allocated']

print("Data cleaning complete! Ready for production load.")
df.head()

Data cleaning complete! Ready for production load.


,Session_ID,Date,Cohort_Quarter,Student_ID,Mentor_ID,STEM_Field,Department_Owner,Budget_Allocated,Budget_Spent,Attendance_Status,Satisfaction_Score,Outcome_Tracked,Student_Gender,Mentor_Gender,Student_Location,Mentor_School,Student_Race,Mentor_Race,Budget_Overrun
0,SES-2026-001,2026-11-24,Q4-2026,STU-118,MNT-524,Robotics,Operations,300,294,Attended,2.0,Landed Internship,Non-binary,Male,Soshanguve,UCT,Coloured,White,False
1,SES-2026-002,2026-10-30,Q4-2026,STU-165,MNT-520,Robotics,Curriculum,900,892,Attended,2.0,Landed Internship,Male,Female,Joburg,UP,Asian,Asian,False
2,SES-2026-003,2026-11-29,Q4-2026,STU-101,MNT-525,Software Engineering,Outreach,1500,1361,Attended,4.0,Retained in STEM,Non-binary,Male,Joburg,UP,Asian,Coloured,False
3,SES-2026-004,2026-06-24,Q2-2026,STU-112,MNT-513,Robotics,Finance,700,661,Attended,3.0,Completed Project,Non-binary,Female,Mamelodi,UJ,Asian,Indian,False
4,SES-2026-005,2026-06-26,Q2-2026,STU-169,MNT-504,Data Science,Operations,1200,1133,Absent,3.0,Ongoing,Male,Female,Pretoria,WITS,Asian,Asian,False


In [5]:
# Select the empty production sheet
clean_sheet = spreadsheet.worksheet('Clean_Set')

# Clear any old data in the production sheet just in case
clean_sheet.clear()

import numpy as np
# Fill any remaining NaN values with None, (Google Sheets API doesn't support NaN)
df_filled = df.replace({np.nan: None})

# Prepare data for Google Sheets update (including headers)
set_with_dataframe_data = [df_filled.columns.values.tolist()] + df_filled.values.tolist()

# Write the clean data
clean_sheet.update(set_with_dataframe_data)
print("Successfully loaded clean data into Clean_Set!")

Successfully loaded clean data into Clean_Set!
